# **Video Object Detection Using MobileNet SSD**

Importing Libraries

In [1]:
import cv2
import numpy as np
from google.colab import files

In [2]:
class_names = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus',
    'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse',
    'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train',
    'tvmonitor'
]

Model Loading

In [5]:
proto = '/content/deploy.prototxt'
model = '/content/mobilenet_iter_73000.caffemodel'

In [6]:
net = cv2.dnn.readNetFromCaffe(proto, model)
print("MobileNet SSD model loaded")

MobileNet SSD model loaded


Video Uploading

In [7]:
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print("Video uploaded:", video_path)

Saving video-for-detection.mp4 to video-for-detection.mp4
Video uploaded: video-for-detection.mp4


In [24]:
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
print("Frames per second:", fps)

Frames per second: 23.976023976023978


In [25]:
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

In [26]:
out_path = "detected_output.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(out_path, fourcc, fps, (W, H))
conf_thresh = 0.5

Model Applying

In [27]:
while True:
    ret, frame = cap.read()
    if not ret:
        break
    blob = cv2.dnn.blobFromImage(frame, 0.007843, (300, 300), 127.5)
    net.setInput(blob)
    detections = net.forward()

    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]
        if confidence > conf_thresh:
            class_id = int(detections[0, 0, i, 1])
            box = detections[0, 0, i, 3:7] * np.array([W, H, W, H])
            x1, y1, x2, y2 = box.astype(int)
            # clamp to frame edges
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(W - 1, x2), min(H - 1, y2)
            label = f"{class_names[class_id]}:{confidence:.2f}"
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, label, (x1, max(20, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    out.write(frame)
cap.release()
out.release()
print("Detection Completed")
files.download("detected_output.mp4")

Detection Completed


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>